### Modeling the spread of rabies in one dimension

In this notebook, we will move from the 2 connected patches described in [Notebook 8](https://github.com/laser-base/laser-generic/blob/50796ab5a253df168255e8a3ac54b766f6aa55d5/notebooks/08_2patch_SIR_wbirths_correlation.ipynb) to a 1-D grid of connected population patches.  This is the scenario explored to describe the spatial spread of rabies in foxes in the work of [Kallen, Arcuri, and Murray, Journal of Theoretical Biology (1985) 116, 377-393](https://pubmed.ncbi.nlm.nih.gov/4058027/). The relevant model equations (note that these are now PDEs rather than the ODEs of previous notebooks) are:
$$
\frac{\partial S}{\partial t} = -KIS \\

\frac{\partial I}{\partial t} = D \frac{\partial^2 I}{\partial x^2} + KIS - \mu I
$$ 

and come to the conclusion that this system supports traveling waves with velocity bounded below by 

$$
c = 2 \sqrt{D \mu (1/r -1)}
$$

where $r = \frac{\mu}{K S_0}$ and $S_0$ is the initial density of susceptibles.

The analogy to the SIR model equations we have been using is clear, though this model conceptually different in specific ways from that model as we now have spatially dimensional quantities.  For example, I & S are infective/susceptible population densities rather than population counts, which affects translating between the $K$ of this model and the $\beta$ of the SIR; the derived quantity $r$ is clearly analogous to $\frac{1}{R_0}$ of the SIR model, but requires the presence of the initial susceptible density to be properly dimensionless, and rabies being invariably fatal, $\mu$ represents the mortality of rabies rather than the $\gamma$ we would use for recovery.  

With all that said, if we are careful about translation between parameters, and appropriately construct the transmission network to appropriately reflect 1-D diffusion, we ought to be able to reproduce this model in our SIR implementation, and recapitulate the traveling wave behavior of the outbreak.

In [1]:
import numpy as np
import pandas as pd
from laser.core.propertyset import PropertySet
import laser.core.distributions as dists
from laser.core.demographics import AliasedDistribution
from laser.core.demographics import KaplanMeierEstimator
from laser_generic.models import SIR
from laser_generic.models.model import Model
from laser_generic.newutils import ValuesMap
from laser_generic.newutils import grid
import laser.core
import laser_generic
import matplotlib.pyplot as plt
import os
from scipy.optimize import fsolve


print(f"{np.__version__=}")
print(f"{laser.core.__version__=}")
print(f"{laser_generic.__version__=}")


np.__version__='2.3.5'
laser.core.__version__='0.6.0'
laser_generic.__version__='0.0.0'


In [2]:
npatches = 201
pop = 10000
scenario = grid(M=1, N=npatches, node_size_km=10, population_fn=lambda x,y: pop, origin_x=0, origin_y=0)
initial_infected = 3
scenario["I"] = 0
scenario["R"] = 0
scenario["S"] = scenario.population
scenario.loc[scenario.nodeid==101,"I"] = initial_infected
scenario.loc[scenario.nodeid==101,"S"] = scenario.loc[scenario.nodeid==101,"population"]-initial_infected
scenario

,nodeid,population,geometry,I,R,S
0,0,10000,"POLYGON ((0 0, 0.08983 0, 0.08983 0.08983, 0 0...",0,0,10000
1,1,10000,"POLYGON ((0.08983 0, 0.17966 0, 0.17966 0.0898...",0,0,10000
2,2,10000,"POLYGON ((0.17966 0, 0.26949 0, 0.26949 0.0898...",0,0,10000
3,3,10000,"POLYGON ((0.26949 0, 0.35932 0, 0.35932 0.0898...",0,0,10000
4,4,10000,"POLYGON ((0.35932 0, 0.44916 0, 0.44916 0.0898...",0,0,10000
...,...,...,...,...,...,...
196,196,10000,"POLYGON ((17.6069 0, 17.69673 0, 17.69673 0.08...",0,0,10000
197,197,10000,"POLYGON ((17.69673 0, 17.78656 0, 17.78656 0.0...",0,0,10000
198,198,10000,"POLYGON ((17.78656 0, 17.87639 0, 17.87639 0.0...",0,0,10000
199,199,10000,"POLYGON ((17.87639 0, 17.96622 0, 17.96622 0.0...",0,0,10000


In [3]:
scenario.loc[101]

nodeid                                                      101
population                                                10000
geometry      POLYGON ((9.072942867409271 0, 9.1627739849083...
I                                                             3
R                                                             0
S                                                          9997
Name: 101, dtype: object

In [ ]:
npatches = 201
pop = 10000
scenario = grid(M=1, N=npatches, node_size_km=10, population_fn=lambda x,y: pop, origin_x=0, origin_y=0)
initial_infected = 3

nticks = 730
R0_samples = 5
infmean_samples = 30

scenario["I"] = 0
scenario["R"] = 0
scenario["S"] = scenario.population

scenario

parameters = PropertySet(
    {
        "seed": np.random.randint(0, 1000000),
        "nticks": nticks,
        "verbose": True,
        "beta": R0 / infmean,
        "inf_mean": infmean,
    }
)

infdurdist = dists.exponential(scale=parameters.inf_mean)
model = Model(scenario, parameters)

model.components = [
    SIR.Susceptible(model),
    SIR.Recovered(model),
    SIR.Infectious(model, infdurdist),
    SIR.Transmission(model, infdurdist),
]
model.run()
outputs[i, :, :] = model.nodes.I
os.makedirs("outputs", exist_ok=True)
np.save(os.path.join("outputs", "CCS_outputs.npy"), outputs)
i+=1
